In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# ============================================================
# 5. PIH-ViTE MODEL + REVIEWER PCB CONTROL
# Corrected Kaggle version
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)


# ------------------------------------------------------------
# Vision Transformer global branch
# ------------------------------------------------------------

class TransformerGlobal(nn.Module):

    def __init__(
        self,
        dim=256,
        depth=6,
        heads=8,
        patch=16
    ):
        super().__init__()

        self.patch = patch

        self.embed = nn.Conv2d(
            in_channels=3,
            out_channels=dim,
            kernel_size=patch,
            stride=patch
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=dim * 4,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

        # Learnable base positional embedding.
        # It will be interpolated when spatial size differs.
        self.pos = nn.Parameter(
            torch.zeros(
                1,
                dim,
                16,
                16
            )
        )

        nn.init.trunc_normal_(
            self.pos,
            std=0.02
        )


    def forward(self, x):

        # Patch embedding
        f = self.embed(x)

        batch_size, channels, height, width = f.shape

        # Resize positional embedding if necessary
        pos = F.interpolate(
            self.pos,
            size=(height, width),
            mode="bicubic",
            align_corners=False
        )

        f = f + pos

        # B,C,H,W -> B,N,C
        tokens = (
            f.flatten(2)
             .transpose(1, 2)
        )

        tokens = self.encoder(tokens)

        # B,N,C -> B,C,H,W
        f = (
            tokens.transpose(1, 2)
                  .reshape(
                      batch_size,
                      channels,
                      height,
                      width
                  )
        )

        return f


# ------------------------------------------------------------
# EfficientNet-B0 local branch
# ------------------------------------------------------------

class EfficientLocal(nn.Module):

    def __init__(self, pretrained=True):

        super().__init__()

        if pretrained:

            weights = (
                EfficientNet_B0_Weights
                .IMAGENET1K_V1
            )

        else:

            weights = None

        network = efficientnet_b0(
            weights=weights
        )

        self.features = network.features


    def forward(self, x):

        outputs = {}

        z = x

        for i, block in enumerate(
            self.features
        ):

            z = block(z)

            # Approximate EfficientNet-B0 feature hierarchy

            if i == 1:
                outputs["s1"] = z

            elif i == 2:
                outputs["s2"] = z

            elif i == 3:
                outputs["s3"] = z

            elif i == 4:
                outputs["s4"] = z

            elif i == 6:
                outputs["s6"] = z

            elif i == 7:
                outputs["s7"] = z
                break

        return outputs


# ------------------------------------------------------------
# Squeeze-and-Excitation block
# ------------------------------------------------------------

class SEBlock(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):

        super().__init__()

        hidden = max(
            channels // reduction,
            8
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1
        )


    def forward(self, x):

        scale = F.adaptive_avg_pool2d(
            x,
            output_size=1
        )

        scale = F.silu(
            self.fc1(scale)
        )

        scale = torch.sigmoid(
            self.fc2(scale)
        )

        return x * scale


# ------------------------------------------------------------
# Structured Physics Constraint Block
# ------------------------------------------------------------

class StructuredPCB(nn.Module):

    def __init__(
        self,
        channels=256
    ):

        super().__init__()

        # Reflectance-like branch:
        # 256 -> 128 -> 3

        self.reflectance_branch = nn.Sequential(

            nn.Conv2d(
                channels,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                128,
                3,
                kernel_size=1
            )
        )

        # Illumination-like branch:
        # 256 -> 128 -> 1

        self.illumination_branch = nn.Sequential(

            nn.Conv2d(
                channels,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                128,
                1,
                kernel_size=1
            )
        )


    def forward(self, features):

        reflectance = torch.sigmoid(
            self.reflectance_branch(
                features
            )
        )

        illumination = torch.sigmoid(
            self.illumination_branch(
                features
            )
        )

        return (
            reflectance,
            illumination
        )


# ------------------------------------------------------------
# Parameter-matched plain convolution control
# Reviewer-2 requested control
# ------------------------------------------------------------

class PlainMatchedHead(nn.Module):

    def __init__(
        self,
        channels=256
    ):

        super().__init__()

        # Shared convolutional mapping
        # instead of explicit separate
        # reflectance / illumination branches.

        self.shared = nn.Sequential(

            nn.Conv2d(
                channels,
                256,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                256,
                4,
                kernel_size=1
            )
        )


    def forward(self, features):

        prediction = torch.sigmoid(
            self.shared(features)
        )

        # First 3 channels
        reflectance_like = (
            prediction[:, 0:3, :, :]
        )

        # Final 1 channel
        illumination_like = (
            prediction[:, 3:4, :, :]
        )

        return (
            reflectance_like,
            illumination_like
        )


# ------------------------------------------------------------
# Standard two-convolution block
# ------------------------------------------------------------

class ConvBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )


    def forward(self, x):

        return self.block(x)


# ------------------------------------------------------------
# Decoder stage
# ------------------------------------------------------------

class DecoderStage(nn.Module):

    def __init__(
        self,
        input_channels,
        skip_channels,
        output_channels
    ):

        super().__init__()

        self.block = ConvBlock(
            input_channels
            + skip_channels,
            output_channels
        )


    def forward(
        self,
        x,
        skip
    ):

        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        x = torch.cat(
            [x, skip],
            dim=1
        )

        return self.block(x)


# ------------------------------------------------------------
# Complete PIH-ViTE
# ------------------------------------------------------------

class PIHViTE(nn.Module):

    def __init__(
        self,
        pcb_mode="structured",
        use_vit=True,
        use_eff=True,
        pretrained_eff=True
    ):

        super().__init__()

        self.use_vit = use_vit
        self.use_eff = use_eff

        # Global branch

        if use_vit:

            self.vit = (
                TransformerGlobal()
            )

        else:

            self.vit = None

        # Local branch

        if use_eff:

            self.eff = EfficientLocal(
                pretrained=pretrained_eff
            )

        else:

            self.eff = None


        # Fusion channel count

        if (
            use_vit
            and use_eff
        ):

            fusion_channels = (
                256 + 320
            )

        elif use_vit:

            fusion_channels = 256

        elif use_eff:

            fusion_channels = 320

        else:

            raise ValueError(
                "At least one encoder "
                "must be enabled."
            )


        # Feature projection

        self.projection = nn.Sequential(

            nn.Conv2d(
                fusion_channels,
                256,
                kernel_size=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(
                inplace=True
            ),

            SEBlock(256)
        )


        # PCB mode

        self.pcb_mode = pcb_mode

        if pcb_mode == "structured":

            self.pcb = StructuredPCB(
                channels=256
            )

        elif pcb_mode == "plain":

            self.pcb = PlainMatchedHead(
                channels=256
            )

        elif pcb_mode == "none":

            self.pcb = None

        else:

            raise ValueError(
                "pcb_mode must be "
                "'structured', "
                "'plain', or 'none'."
            )


        # Decoder

        self.bottleneck_decoder = (
            ConvBlock(
                256 + 192,
                256
            )
        )

        self.decoder4 = DecoderStage(
            256,
            80,
            160
        )

        self.decoder3 = DecoderStage(
            160,
            40,
            96
        )

        self.decoder2 = DecoderStage(
            96,
            24,
            64
        )

        self.decoder1 = DecoderStage(
            64,
            16,
            32
        )


        self.output_head = nn.Sequential(

            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                32,
                3,
                kernel_size=1
            ),

            nn.Sigmoid()
        )


    def forward(self, x):

        batch_size = x.shape[0]

        image_height = x.shape[2]
        image_width = x.shape[3]

        # --------------------------------
        # EfficientNet features
        # --------------------------------

        if self.eff is not None:

            local_features = (
                self.eff(x)
            )

        else:

            local_features = None


        # --------------------------------
        # ViT features
        # --------------------------------

        if self.vit is not None:

            global_features = (
                self.vit(x)
            )

        else:

            global_features = None


        # --------------------------------
        # Feature fusion
        # --------------------------------

        if (
            self.use_vit
            and self.use_eff
        ):

            global_features = (
                F.interpolate(
                    global_features,
                    size=local_features[
                        "s7"
                    ].shape[-2:],
                    mode="area"
                )
            )

            fused = torch.cat(

                [
                    global_features,
                    local_features["s7"]
                ],

                dim=1
            )


        elif self.use_vit:

            fused = global_features


        else:

            fused = (
                local_features["s7"]
            )


        bottleneck = (
            self.projection(
                fused
            )
        )


        # --------------------------------
        # Physics constraint
        # --------------------------------

        reflectance = None
        illumination = None

        if self.pcb is not None:

            (
                reflectance,
                illumination
            ) = self.pcb(
                bottleneck
            )


        # --------------------------------
        # Decoder skip features
        # --------------------------------

        if local_features is not None:

            s6 = local_features["s6"]
            s4 = local_features["s4"]
            s3 = local_features["s3"]
            s2 = local_features["s2"]
            s1 = local_features["s1"]


        else:

            # Zero skip tensors are used
            # only for ViT-only ablation.

            def zero_feature(
                channels,
                downsample_factor
            ):

                height = max(
                    1,
                    image_height
                    // downsample_factor
                )

                width = max(
                    1,
                    image_width
                    // downsample_factor
                )

                return torch.zeros(

                    batch_size,
                    channels,
                    height,
                    width,

                    device=x.device,
                    dtype=x.dtype
                )


            s6 = zero_feature(
                192,
                32
            )

            s4 = zero_feature(
                80,
                16
            )

            s3 = zero_feature(
                40,
                8
            )

            s2 = zero_feature(
                24,
                4
            )

            s1 = zero_feature(
                16,
                2
            )


        # Ensure bottleneck resolution
        # matches deepest decoder skip

        if (
            bottleneck.shape[-2:]
            !=
            s6.shape[-2:]
        ):

            bottleneck = (
                F.interpolate(
                    bottleneck,
                    size=s6.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                )
            )


        decoder = (
            self.bottleneck_decoder(
                torch.cat(
                    [
                        bottleneck,
                        s6
                    ],
                    dim=1
                )
            )
        )


        decoder = self.decoder4(
            decoder,
            s4
        )

        decoder = self.decoder3(
            decoder,
            s3
        )

        decoder = self.decoder2(
            decoder,
            s2
        )

        decoder = self.decoder1(
            decoder,
            s1
        )


        # Recover original resolution

        decoder = F.interpolate(

            decoder,

            size=(
                image_height,
                image_width
            ),

            mode="bilinear",

            align_corners=False
        )


        enhanced = (
            self.output_head(
                decoder
            )
        )


        return {

            "image":
                enhanced,

            "R":
                reflectance,

            "L":
                illumination
        }


# ------------------------------------------------------------
# Parameter-count utility
# ------------------------------------------------------------

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
    )


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

structured_model = PIHViTE(
    pcb_mode="structured",
    pretrained_eff=False
)

plain_model = PIHViTE(
    pcb_mode="plain",
    pretrained_eff=False
)


print(
    "Full structured model parameters:",
    f"{count_parameters(structured_model):,}"
)

print(
    "Full plain-control model parameters:",
    f"{count_parameters(plain_model):,}"
)

print(
    "Structured PCB parameters:",
    f"{count_parameters(structured_model.pcb):,}"
)

print(
    "Plain head parameters:",
    f"{count_parameters(plain_model.pcb):,}"
)


pcb_params = count_parameters(
    structured_model.pcb
)

plain_params = count_parameters(
    plain_model.pcb
)

difference_percent = (
    100.0
    *
    (
        plain_params
        -
        pcb_params
    )
    /
    pcb_params
)

print(
    "Head parameter difference:",
    f"{difference_percent:.2f}%"
)


# ------------------------------------------------------------
# Forward-pass test
# ------------------------------------------------------------

dummy = torch.randn(
    1,
    3,
    256,
    256
)

with torch.no_grad():

    output = structured_model(
        dummy
    )


print(
    "\nForward test successful."
)

print(
    "Enhanced image:",
    output["image"].shape
)

print(
    "Reflectance:",
    output["R"].shape
)

print(
    "Illumination:",
    output["L"].shape
)


del structured_model
del plain_model
del dummy
del output

print(
    "\n✓ MODEL CELL PASSED"
)

/tmp/ipykernel_58/651200585.py:50: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


Full structured model parameters: 12,499,539
Full plain-control model parameters: 12,500,051
Structured PCB parameters: 591,108
Plain head parameters: 591,620
Head parameter difference: 0.09%

Forward test successful.
Enhanced image: torch.Size([1, 3, 256, 256])
Reflectance: torch.Size([1, 3, 8, 8])
Illumination: torch.Size([1, 1, 8, 8])

✓ MODEL CELL PASSED


In [4]:
# ============================================================
# 6. PIH-ViTE LOSSES + COMPLETE SANITY CHECK
# Run this AFTER the model cell
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import (
    vgg16,
    VGG16_Weights
)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# VGG perceptual loss
# ------------------------------------------------------------

class VGGPerceptualLoss(nn.Module):

    def __init__(self):

        super().__init__()

        print(
            "Loading pretrained VGG16 "
            "for perceptual loss..."
        )

        try:

            weights = (
                VGG16_Weights
                .IMAGENET1K_V1
            )

            vgg = vgg16(
                weights=weights
            ).features

        except Exception as error:

            print(
                "\nCould not load pretrained "
                "VGG16 weights."
            )

            print(
                "Make sure Kaggle Internet "
                "is enabled."
            )

            raise error


        # Up to relu2_2 region

        self.block1 = nn.Sequential(
            *list(
                vgg.children()
            )[:9]
        )


        # relu2_2 -> relu3_3 region

        self.block2 = nn.Sequential(
            *list(
                vgg.children()
            )[9:16]
        )


        for parameter in self.parameters():

            parameter.requires_grad = False


        self.eval()


        self.register_buffer(

            "mean",

            torch.tensor(
                [
                    0.485,
                    0.456,
                    0.406
                ]
            ).view(
                1,
                3,
                1,
                1
            )
        )


        self.register_buffer(

            "std",

            torch.tensor(
                [
                    0.229,
                    0.224,
                    0.225
                ]
            ).view(
                1,
                3,
                1,
                1
            )
        )


    def normalize(self, x):

        return (
            x - self.mean
        ) / self.std


    def forward(
        self,
        prediction,
        target
    ):

        prediction = self.normalize(
            prediction
        )

        target = self.normalize(
            target
        )


        pred_1 = self.block1(
            prediction
        )

        target_1 = self.block1(
            target
        )


        pred_2 = self.block2(
            pred_1
        )

        target_2 = self.block2(
            target_1
        )


        loss_1 = F.l1_loss(
            pred_1,
            target_1
        )


        loss_2 = F.l1_loss(
            pred_2,
            target_2
        )


        return (
            loss_1
            +
            loss_2
        )


# ------------------------------------------------------------
# SSIM loss
# ------------------------------------------------------------

def ssim_loss(
    prediction,
    target
):

    C1 = 0.01 ** 2

    C2 = 0.03 ** 2

    kernel_size = 11

    padding = (
        kernel_size
        //
        2
    )


    mu_x = F.avg_pool2d(

        prediction,

        kernel_size,

        stride=1,

        padding=padding
    )


    mu_y = F.avg_pool2d(

        target,

        kernel_size,

        stride=1,

        padding=padding
    )


    sigma_x = (

        F.avg_pool2d(

            prediction
            *
            prediction,

            kernel_size,

            stride=1,

            padding=padding
        )

        -

        mu_x
        *
        mu_x
    )


    sigma_y = (

        F.avg_pool2d(

            target
            *
            target,

            kernel_size,

            stride=1,

            padding=padding
        )

        -

        mu_y
        *
        mu_y
    )


    sigma_xy = (

        F.avg_pool2d(

            prediction
            *
            target,

            kernel_size,

            stride=1,

            padding=padding
        )

        -

        mu_x
        *
        mu_y
    )


    numerator = (

        (
            2
            *
            mu_x
            *
            mu_y
            +
            C1
        )

        *

        (
            2
            *
            sigma_xy
            +
            C2
        )
    )


    denominator = (

        (
            mu_x
            *
            mu_x

            +

            mu_y
            *
            mu_y

            +

            C1
        )

        *

        (
            sigma_x

            +

            sigma_y

            +

            C2
        )
    )


    ssim_map = (

        numerator

        /

        (
            denominator
            +
            1e-8
        )
    )


    return (

        1.0

        -

        ssim_map.mean()
    )


# ------------------------------------------------------------
# Total variation loss
# ------------------------------------------------------------

def total_variation_loss(
    illumination
):

    vertical = (

        illumination[
            :,
            :,
            1:,
            :
        ]

        -

        illumination[
            :,
            :,
            :-1,
            :
        ]

    ).abs().mean()


    horizontal = (

        illumination[
            :,
            :,
            :,
            1:
        ]

        -

        illumination[
            :,
            :,
            :,
            :-1
        ]

    ).abs().mean()


    return (
        vertical
        +
        horizontal
    )


# ------------------------------------------------------------
# Retinex product consistency
# ------------------------------------------------------------

def retinex_consistency_loss(
    reflectance,
    illumination,
    target
):

    target_size = (
        target.shape[-2:]
    )


    reflectance_up = F.interpolate(

        reflectance,

        size=target_size,

        mode="bilinear",

        align_corners=False
    )


    illumination_up = F.interpolate(

        illumination,

        size=target_size,

        mode="bilinear",

        align_corners=False
    )


    reconstructed = (

        reflectance_up
        *
        illumination_up
    )


    return F.l1_loss(

        reconstructed,

        target
    )


# ------------------------------------------------------------
# Complete compound loss
# ------------------------------------------------------------

class PIHViTELoss(nn.Module):

    def __init__(
        self,
        lambda_l1=1.0,
        lambda_perceptual=0.1,
        lambda_ssim=0.2,
        lambda_tv=1e-4,
        lambda_retinex=0.05
    ):

        super().__init__()


        self.lambda_l1 = (
            lambda_l1
        )

        self.lambda_perceptual = (
            lambda_perceptual
        )

        self.lambda_ssim = (
            lambda_ssim
        )

        self.lambda_tv = (
            lambda_tv
        )

        self.lambda_retinex = (
            lambda_retinex
        )


        self.perceptual = (
            VGGPerceptualLoss()
        )


    def forward(
        self,
        model_output,
        target,
        use_tv=True,
        use_retinex=True
    ):

        prediction = (
            model_output[
                "image"
            ]
        )


        reflectance = (
            model_output[
                "R"
            ]
        )


        illumination = (
            model_output[
                "L"
            ]
        )


        # --------------------------------
        # L1
        # --------------------------------

        loss_l1 = F.l1_loss(

            prediction,

            target
        )


        # --------------------------------
        # Perceptual
        # --------------------------------

        loss_perceptual = (

            self.perceptual(

                prediction,

                target
            )
        )


        # --------------------------------
        # SSIM
        # --------------------------------

        loss_ssim = ssim_loss(

            prediction,

            target
        )


        # --------------------------------
        # Physics losses
        # --------------------------------

        loss_tv = torch.zeros(

            (),

            device=prediction.device
        )


        loss_retinex = torch.zeros(

            (),

            device=prediction.device
        )


        if (

            illumination
            is not None

            and

            use_tv
        ):

            loss_tv = (
                total_variation_loss(
                    illumination
                )
            )


        if (

            reflectance
            is not None

            and

            illumination
            is not None

            and

            use_retinex
        ):

            loss_retinex = (

                retinex_consistency_loss(

                    reflectance,

                    illumination,

                    target
                )
            )


        total_loss = (

            self.lambda_l1
            *
            loss_l1

            +

            self.lambda_perceptual
            *
            loss_perceptual

            +

            self.lambda_ssim
            *
            loss_ssim

            +

            self.lambda_tv
            *
            loss_tv

            +

            self.lambda_retinex
            *
            loss_retinex
        )


        components = {

            "total":
                total_loss
                .detach()
                .item(),

            "l1":
                loss_l1
                .detach()
                .item(),

            "perceptual":
                loss_perceptual
                .detach()
                .item(),

            "ssim":
                loss_ssim
                .detach()
                .item(),

            "tv":
                loss_tv
                .detach()
                .item(),

            "retinex":
                loss_retinex
                .detach()
                .item()
        }


        return (
            total_loss,
            components
        )


# ============================================================
# SANITY TEST
# ============================================================

print(
    "\nCreating PIH-ViTE "
    "sanity-test model..."
)


test_model = PIHViTE(

    pcb_mode="structured",

    pretrained_eff=False

).to(
    DEVICE
)


criterion = PIHViTELoss().to(
    DEVICE
)


# Use one 256x256 sample,
# exactly matching training crop size.

dummy_input = torch.rand(

    1,
    3,
    256,
    256,

    device=DEVICE
)


dummy_target = torch.rand(

    1,
    3,
    256,
    256,

    device=DEVICE
)


test_model.train()


print(
    "Running forward pass..."
)


output = test_model(
    dummy_input
)


print(
    "Output image:",
    output[
        "image"
    ].shape
)


print(
    "Reflectance:",
    output[
        "R"
    ].shape
)


print(
    "Illumination:",
    output[
        "L"
    ].shape
)


print(
    "\nComputing losses..."
)


loss, components = criterion(

    output,

    dummy_target,

    use_tv=True,

    use_retinex=True
)


print(
    "\nLoss components"
)


for key, value in components.items():

    print(
        f"{key:12s}: "
        f"{value:.6f}"
    )


# ------------------------------------------------------------
# Numerical checks
# ------------------------------------------------------------

assert torch.isfinite(
    loss
), "Loss contains NaN or Inf"


for key, value in components.items():

    assert float(
        value
    ) == float(
        value
    ), f"{key} is NaN"


# ------------------------------------------------------------
# Backpropagation test
# ------------------------------------------------------------

print(
    "\nRunning backward pass..."
)


loss.backward()


trainable_parameters = [

    parameter

    for parameter
    in test_model.parameters()

    if parameter.requires_grad
]


parameters_with_gradient = [

    parameter

    for parameter
    in trainable_parameters

    if parameter.grad
    is not None
]


print(
    "Trainable tensors:",
    len(
        trainable_parameters
    )
)


print(
    "Tensors with gradients:",
    len(
        parameters_with_gradient
    )
)


assert (
    len(
        parameters_with_gradient
    )
    >
    0
)


# Check PCB specifically

pcb_gradients = [

    parameter.grad

    for parameter
    in test_model.pcb.parameters()

    if parameter.grad
    is not None
]


print(
    "PCB tensors with gradients:",
    len(
        pcb_gradients
    )
)


assert (
    len(
        pcb_gradients
    )
    >
    0
), "PCB is not receiving gradients"


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del output
del dummy_input
del dummy_target
del test_model
del criterion


if torch.cuda.is_available():

    torch.cuda.empty_cache()


print(
    "\n================================"
)

print(
    "✓ LOSS DEFINITION PASSED"
)

print(
    "✓ FORWARD PASS PASSED"
)

print(
    "✓ RETINEX LOSS PASSED"
)

print(
    "✓ TV LOSS PASSED"
)

print(
    "✓ BACKWARD PASS PASSED"
)

print(
    "✓ PCB GRADIENT CHECK PASSED"
)

print(
    "================================"
)

Device: cpu

Creating PIH-ViTE sanity-test model...
Loading pretrained VGG16 for perceptual loss...


/tmp/ipykernel_58/651200585.py:50: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 147MB/s] 


Running forward pass...
Output image: torch.Size([1, 3, 256, 256])
Reflectance: torch.Size([1, 3, 8, 8])
Illumination: torch.Size([1, 1, 8, 8])

Computing losses...

Loss components
total       : 0.808714
l1          : 0.250851
perceptual  : 3.522926
ssim        : 0.945012
tv          : 0.175281
retinex     : 0.331004

Running backward pass...
Trainable tensors: 350
Tensors with gradients: 347
PCB tensors with gradients: 12

✓ LOSS DEFINITION PASSED
✓ FORWARD PASS PASSED
✓ RETINEX LOSS PASSED
✓ TV LOSS PASSED
✓ BACKWARD PASS PASSED
✓ PCB GRADIENT CHECK PASSED


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )

    DEVICE = torch.device("cuda:0")

else:
    DEVICE = torch.device("cpu")

print("Selected device:", DEVICE)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Selected device: cuda:0


In [3]:
# ============================================================
# RECOVER LOL-v1 DATASET STATE
# Creates train_pairs and test_pairs again
# ============================================================

from pathlib import Path
from PIL import Image
import zipfile
import os

IMG_EXTS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
}

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/LOLv1")

WORK_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Extract archive if necessary
# ------------------------------------------------------------

def contains_lol_structure(root):

    names = [
        str(p).lower()
        for p in root.rglob("*")
        if p.is_dir()
    ]

    has_our485 = any(
        "our485" in x
        for x in names
    )

    has_eval15 = any(
        "eval15" in x
        for x in names
    )

    return (
        has_our485
        and
        has_eval15
    )


if not contains_lol_structure(
    WORK_ROOT
):

    zip_files = list(
        KAGGLE_INPUT.rglob(
            "*.zip"
        )
    )

    print(
        "ZIP files found:",
        len(zip_files)
    )

    for z in zip_files[:10]:

        print(
            " -",
            z
        )


    if zip_files:

        archive = None

        # Prefer archive.zip

        for z in zip_files:

            if (
                z.name.lower()
                ==
                "archive.zip"
            ):

                archive = z

                break


        if archive is None:

            archive = zip_files[0]


        print(
            "\nExtracting:",
            archive
        )


        with zipfile.ZipFile(
            archive,
            "r"
        ) as zf:

            zf.extractall(
                WORK_ROOT
            )


        print(
            "Extraction complete."
        )


# ------------------------------------------------------------
# 2. Find folders robustly
# ------------------------------------------------------------

def find_folder(
    search_root,
    ending_parts
):

    ending_parts = [
        part.lower()
        for part
        in ending_parts
    ]


    for path in search_root.rglob(
        "*"
    ):

        if not path.is_dir():

            continue


        parts = [
            p.lower()
            for p
            in path.parts
        ]


        if (
            len(parts)
            >=
            len(ending_parts)
        ):

            if (
                parts[
                    -len(
                        ending_parts
                    ):
                ]
                ==
                ending_parts
            ):

                return path


    return None


SEARCH_ROOTS = [

    WORK_ROOT,

    KAGGLE_INPUT
]


TRAIN_LOW = None
TRAIN_HIGH = None
TEST_LOW = None
TEST_HIGH = None


for root in SEARCH_ROOTS:

    if TRAIN_LOW is None:

        TRAIN_LOW = find_folder(
            root,
            [
                "our485",
                "low"
            ]
        )


    if TRAIN_HIGH is None:

        TRAIN_HIGH = find_folder(
            root,
            [
                "our485",
                "high"
            ]
        )


    if TEST_LOW is None:

        TEST_LOW = find_folder(
            root,
            [
                "eval15",
                "low"
            ]
        )


    if TEST_HIGH is None:

        TEST_HIGH = find_folder(
            root,
            [
                "eval15",
                "high"
            ]
        )


print(
    "\nDetected folders"
)

print(
    "TRAIN_LOW :",
    TRAIN_LOW
)

print(
    "TRAIN_HIGH:",
    TRAIN_HIGH
)

print(
    "TEST_LOW  :",
    TEST_LOW
)

print(
    "TEST_HIGH :",
    TEST_HIGH
)


assert TRAIN_LOW is not None, (
    "Could not locate our485/low"
)

assert TRAIN_HIGH is not None, (
    "Could not locate our485/high"
)

assert TEST_LOW is not None, (
    "Could not locate eval15/low"
)

assert TEST_HIGH is not None, (
    "Could not locate eval15/high"
)


# ------------------------------------------------------------
# 3. List image files
# ------------------------------------------------------------

def get_images(
    folder
):

    return sorted(

        [
            p

            for p
            in folder.iterdir()

            if (
                p.is_file()

                and

                p.suffix.lower()
                in
                IMG_EXTS
            )
        ]
    )


train_low_files = get_images(
    TRAIN_LOW
)

train_high_files = get_images(
    TRAIN_HIGH
)

test_low_files = get_images(
    TEST_LOW
)

test_high_files = get_images(
    TEST_HIGH
)


print(
    "\nRaw image counts"
)

print(
    "Training low :",
    len(
        train_low_files
    )
)

print(
    "Training high:",
    len(
        train_high_files
    )
)

print(
    "Testing low  :",
    len(
        test_low_files
    )
)

print(
    "Testing high :",
    len(
        test_high_files
    )
)


# ------------------------------------------------------------
# 4. Create paired images
# ------------------------------------------------------------

def create_pairs(
    low_files,
    high_files
):

    low_by_name = {

        p.name:
        p

        for p
        in low_files
    }


    high_by_name = {

        p.name:
        p

        for p
        in high_files
    }


    common_names = sorted(

        set(
            low_by_name.keys()
        )

        &

        set(
            high_by_name.keys()
        )
    )


    # If filenames differ in extensions,
    # retry using stems.

    if not common_names:

        low_by_stem = {

            p.stem:
            p

            for p
            in low_files
        }


        high_by_stem = {

            p.stem:
            p

            for p
            in high_files
        }


        common_stems = sorted(

            set(
                low_by_stem.keys()
            )

            &

            set(
                high_by_stem.keys()
            )
        )


        pairs = [

            (
                low_by_stem[
                    name
                ],

                high_by_stem[
                    name
                ]
            )

            for name
            in common_stems
        ]


    else:

        pairs = [

            (
                low_by_name[
                    name
                ],

                high_by_name[
                    name
                ]
            )

            for name
            in common_names
        ]


    return pairs


train_pairs = create_pairs(

    train_low_files,

    train_high_files
)


test_pairs = create_pairs(

    test_low_files,

    test_high_files
)


# ------------------------------------------------------------
# 5. Verify official protocol
# ------------------------------------------------------------

print(
    "\nPaired dataset"
)

print(
    "train_pairs:",
    len(
        train_pairs
    )
)

print(
    "test_pairs :",
    len(
        test_pairs
    )
)


assert len(train_pairs) == 485, (

    "Official LOL-v1 requires "
    "485 training pairs, but "
    f"{len(train_pairs)} were found."
)


assert len(test_pairs) == 15, (

    "Official LOL-v1 requires "
    "15 test pairs, but "
    f"{len(test_pairs)} were found."
)


# ------------------------------------------------------------
# 6. Verify several pair dimensions
# ------------------------------------------------------------

print(
    "\nChecking image pairs..."
)


for index, (
    low_path,
    high_path
) in enumerate(
    train_pairs[:5]
):

    low_img = Image.open(
        low_path
    )

    high_img = Image.open(
        high_path
    )


    print(

        index,

        low_path.name,

        "Low:",
        low_img.size,

        "GT:",
        high_img.size
    )


    assert (
        low_img.size
        ==
        high_img.size
    )


print(
    "\n================================"
)

print(
    "✓ OFFICIAL LOL-v1 RECOVERED"
)

print(
    "✓ 485 TRAINING PAIRS FOUND"
)

print(
    "✓ 15 TEST PAIRS FOUND"
)

print(
    "✓ train_pairs CREATED"
)

print(
    "✓ test_pairs CREATED"
)

print(
    "================================"
)

ZIP files found: 0

Detected folders
TRAIN_LOW : None
TRAIN_HIGH: None
TEST_LOW  : None
TEST_HIGH : None


AssertionError: Could not locate our485/low